# Assignment 25 — Prompting & LangChain Chains

**Student:** Abhishek Thakare

This one builds on top of Assignment 22. Back then I was only loading documents,
chunking them, and testing embeddings + vector search. This time the goal is to
actually *do something* with an LLM on top of that - prompt templates, structured
output, chains, and LCEL.

I'm reusing the same knowledge base from Assignment 22 (the onboarding notes +
FAQ csv for the Personal Knowledge Assistant project), so the vector store part
of this notebook is basically the same setup, just wired into a chat model now.

**A quick honest note on the LLM:** I still don't have working OpenAI credits
(same issue as last time), so everywhere this notebook needs a chat model I'm
using **Ollama running `llama3.2` locally**. For embeddings I'm sticking with
the same Hugging Face model I used in Assignment 22
(`sentence-transformers/all-MiniLM-L6-v2`) since that one actually worked without
any API key.


## Before running this

Files this notebook expects (same folder as Assignment 22):

```text
data/
├── notes.txt
└── data.csv
```

Also needed, running locally:
- Ollama running, with `llama3.2` pulled (`ollama pull llama3.2`)
- `nomic-embed-text` isn't needed here, I switched to the Hugging Face embedding
  model instead since it's simpler to keep everything local in one place.

If Ollama isn't running when a cell tries to call it, I catch the connection
error and print that instead of pretending it worked.


In [13]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-ollama faiss-cpu sentence-transformers pydantic

In [14]:
# quick sanity check on the two files this notebook needs
import os
assert os.path.exists("data/notes.txt"), "Missing data/notes.txt"
assert os.path.exists("data/data.csv"), "Missing data/data.csv"
print("Both data files found.")

Both data files found.


In [15]:
import os
import json

print("Setup check")
print("-" * 40)
print("Data folder present:", os.path.isdir("data"))


Setup check
----------------------------------------
Data folder present: True


## PART 1 — Prompt Templates

### Task 1: PromptTemplate

Simplest possible version first - one system-style instruction baked into the
template, and one placeholder for whatever the employee actually asks.


In [16]:
from langchain_core.prompts import PromptTemplate

simple_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are the Personal Knowledge Assistant for new employees.\n"
        "Answer the question clearly, using simple language.\n\n"
        "Question: {question}\n"
        "Answer:"
    ),
)

# testing with a few real onboarding questions someone would actually ask
test_questions = [
    "How many paid leaves do I get?",
    "Who do I contact if my laptop isn't working?",
    "What's for lunch today?",  # just to see how it handles something off-topic
]

for q in test_questions:
    print("-" * 60)
    print(simple_prompt.format(question=q))


------------------------------------------------------------
You are the Personal Knowledge Assistant for new employees.
Answer the question clearly, using simple language.

Question: How many paid leaves do I get?
Answer:
------------------------------------------------------------
You are the Personal Knowledge Assistant for new employees.
Answer the question clearly, using simple language.

Question: Who do I contact if my laptop isn't working?
Answer:
------------------------------------------------------------
You are the Personal Knowledge Assistant for new employees.
Answer the question clearly, using simple language.

Question: What's for lunch today?
Answer:


Nothing fancy here - `simple_prompt.format(...)` just drops the question into
the template string. It works fine for a single quick question, but there's no
real way to separate "instructions for the assistant" from "what the user said" -
it's all one flat string. That's exactly what Task 2 fixes.


### Task 2: ChatPromptTemplate & Message Templates

Same idea, but built out of actual message objects instead of one string.
I added a short `AIMessage` example too, mostly to nudge the tone of the reply.


In [17]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)

system_msg = SystemMessagePromptTemplate.from_template(
    "You are the Personal Knowledge Assistant. You answer onboarding questions "
    "using the company's HR and IT documents. If you're not sure, say so instead "
    "of guessing."
)

# a one-line example turn, just so the model has a sense of the tone I want
ai_example = AIMessagePromptTemplate.from_template(
    "Sure, here's what I found in the onboarding docs:"
)

human_msg = HumanMessagePromptTemplate.from_template("{question}")

chat_prompt = ChatPromptTemplate.from_messages([system_msg, ai_example, human_msg])

rendered = chat_prompt.format_messages(question="How many paid leaves do I get?")
for msg in rendered:
    print(f"[{msg.type}] {msg.content}")


[system] You are the Personal Knowledge Assistant. You answer onboarding questions using the company's HR and IT documents. If you're not sure, say so instead of guessing.
[ai] Sure, here's what I found in the onboarding docs:
[human] How many paid leaves do I get?


### Comparing the two

`PromptTemplate` gives me one plain string - good enough for a single-shot
question. `ChatPromptTemplate` gives me a *list* of role-tagged messages
(system / ai / human), which is what chat models are actually built to use.
The system message stays fixed no matter what the employee asks, and only the
human message changes - that separation is the whole point, and it's why every
chain later in this notebook uses `ChatPromptTemplate` instead of the plain one.


## Setting up the knowledge base (reused from Assignment 22)

This part is basically copy-pasted from Assignment 22: load the two files,
split them into chunks, embed them, and store them in FAISS so I have a
retriever ready for Task 6 and Task 9 later.


In [18]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

txt_docs = TextLoader("data/notes.txt").load()
csv_docs = CSVLoader("data/data.csv").load()
all_docs = txt_docs + csv_docs

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)

print("Loaded documents:", len(all_docs))
print("Chunks after splitting:", len(chunks))


Loaded documents: 9
Chunks after splitting: 14


In [19]:
# Same embedding model as Assignment 22 - it's local, no API key needed,
# and I already know it works.
retriever = None
faiss_db = None

try:
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import FAISS

    hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    faiss_db = FAISS.from_documents(chunks, hf_embeddings)
    retriever = faiss_db.as_retriever(search_kwargs={"k": 3})
    print("Vector store ready. Quick test search:")
    for doc in retriever.invoke("What is the leave policy?"):
        print("-", doc.page_content[:120].replace("\n", " "))
except Exception as e:
    print("Could not build the vector store:", e)
    print("(This step needs internet access the first time, to download the embedding model.)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready. Quick test search:
- employee_id: 107 name: Meera Joshi department: Data role: Data Analyst years_experience: 2 location: Mumbai
- employee_id: 102 name: Rahul Verma department: Engineering role: ML Engineer years_experience: 5 location: Bengaluru
- employee_id: 106 name: Arjun Reddy department: Engineering role: Backend Developer years_experience: 6 location: Bengalu


## PART 2 — Structured Output using Pydantic

### Task 3: Pydantic Output Schema

Right now the chatbot just returns whatever text it feels like. If I ever want
to show this in an actual UI (a confidence score, a "source" tag, etc.) I need
the output in a fixed shape every single time, not a paragraph I have to guess
at parsing. That's what this schema is for.


In [20]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Answer(BaseModel):
    answer: str = Field(description="the direct answer to the employee's question")
    confidence: float = Field(description="how confident the model is, from 0.0 to 1.0", ge=0.0, le=1.0)
    source: str = Field(description="where the answer came from, e.g. 'onboarding_docs' or 'general_knowledge'")

answer_parser = PydanticOutputParser(pydantic_object=Answer)
print(answer_parser.get_format_instructions())


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"description": "the direct answer to the employee's question", "title": "Answer", "type": "string"}, "confidence": {"description": "how confident the model is, from 0.0 to 1.0", "maximum": 1.0, "minimum": 0.0, "title": "Confidence", "type": "number"}, "source": {"description": "where the answer came from, e.g. 'onboarding_docs' or 'general_knowledge'", "title": "Source", "type": "string"}}, "required": ["answer", "confidence", "source"]}
```


That printed block is what actually gets added to the prompt so the model
knows exactly what JSON shape to reply with. If it follows it properly,
`answer_parser.parse(raw_text)` turns the raw string into a real `Answer` object
I can use in code (`result.confidence`, `result.source`, etc.) instead of
scraping a paragraph.


### Task 4: Validation & Error Handling

LLMs don't always follow the format perfectly - a missing field, a confidence
of `"high"` instead of a number, or just plain broken JSON. I tested this with
a good response and a couple of broken ones to see what actually happens.


In [21]:
good_output = '{"answer": "18 paid leaves per year", "confidence": 0.95, "source": "onboarding_docs"}'
missing_field_output = '{"answer": "18 paid leaves per year", "confidence": 0.95}'
wrong_type_output = '{"answer": "18 paid leaves per year", "confidence": "very sure", "source": "onboarding_docs"}'

for label, raw in [
    ("good output", good_output),
    ("missing field", missing_field_output),
    ("wrong type for confidence", wrong_type_output),
]:
    try:
        parsed = answer_parser.parse(raw)
        print(f"{label}: parsed fine -> {parsed}")
    except Exception as e:
        print(f"{label}: FAILED to parse -> {type(e).__name__}")


good output: parsed fine -> answer='18 paid leaves per year' confidence=0.95 source='onboarding_docs'
missing field: FAILED to parse -> OutputParserException
wrong type for confidence: FAILED to parse -> OutputParserException


In [22]:
# When the strict parser fails, the usual fix is LangChain's OutputFixingParser -
# it sends the broken output + the error back to an LLM and asks it to repair
# the formatting, then tries parsing again. That needs a working LLM, so here
# I fall back to a safe default instead of crashing the whole pipeline if even
# the fix attempt fails.

def safe_parse(raw_text, llm=None):
    try:
        return answer_parser.parse(raw_text)
    except Exception as strict_error:
        print("Strict parse failed:", strict_error)
        if llm is None:
            print("No LLM passed in to attempt a fix, returning a default answer instead.")
            return Answer(answer="Sorry, I couldn't produce a valid answer.", confidence=0.0, source="fallback_default")
        try:
            from langchain.output_parsers import OutputFixingParser
            fixer = OutputFixingParser.from_llm(parser=answer_parser, llm=llm)
            return fixer.parse(raw_text)
        except Exception as fix_error:
            print("Fix attempt also failed:", fix_error)
            return Answer(answer="Sorry, I couldn't produce a valid answer.", confidence=0.0, source="fallback_default")

print(safe_parse(missing_field_output))


Strict parse failed: Failed to parse Answer from completion {"answer": "18 paid leaves per year", "confidence": 0.95}. Got: 1 validation error for Answer
source
  Field required [type=missing, input_value={'answer': '18 paid leave...ar', 'confidence': 0.95}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
No LLM passed in to attempt a fix, returning a default answer instead.
answer="Sorry, I couldn't produce a valid answer." confidence=0.0 source='fallback_default'


So the strict parser genuinely rejects anything that doesn't match the schema -
that's the whole point of using Pydantic here instead of just eyeballing the
text. Without a real LLM available to fix it, my fallback just returns a safe
default object instead of letting the app crash. With Ollama running, I'd pass
the llm in and let `OutputFixingParser` actually repair it.


## PART 3 — Chains in LangChain

In [23]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = None
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2", temperature=0.2)
    # quick check it's actually reachable
    llm.invoke("say ok")
    print("Ollama is up, llama3.2 responded.")
except Exception as e:
    print("Couldn't reach Ollama:", e)
    print("The chain cells below will just print this same message instead of a real answer.")


Ollama is up, llama3.2 responded.


### Task 5: Simple Chain

Prompt → LLM → Output. This is the basic building block everything else in
this notebook is made of.


In [24]:
simple_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the Personal Knowledge Assistant. Keep answers short."),
    ("human", "{question}"),
])

simple_chain = simple_chat_prompt | llm | StrOutputParser() if llm else None

def run_simple_chain(question):
    if simple_chain is None:
        return "[Ollama not available - can't generate a real answer right now]"
    return simple_chain.invoke({"question": question})

print(run_simple_chain("How many paid leaves do I get?"))


Typically, in the US, you get 10 paid federal holidays and 5-7 paid sick days per year, depending on the company.


### Task 6: Conditional Chain

Some questions need the actual onboarding docs (leave policy, IT contact,
etc.) and some are just casual chat that the model can answer on its own. So
this chain first asks the model to label the question, then routes it: factual
questions go through the retriever, everything else goes straight to the LLM.


In [25]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reply with exactly one word: 'factual' if this needs looked-up company info, "
               "or 'casual' if it's small talk. Nothing else."),
    ("human", "{question}"),
])

factual_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context. If it's not in there, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

casual_prompt = ChatPromptTemplate.from_messages([
    ("system", "You're a friendly assistant, just chat normally."),
    ("human", "{question}"),
])

def is_factual(payload):
    label = (classify_prompt | llm | StrOutputParser()).invoke({"question": payload["question"]})
    return "factual" in label.strip().lower()

def factual_branch(payload):
    docs = retriever.invoke(payload["question"])
    context = "\n\n".join(d.page_content for d in docs)
    return (factual_prompt | llm | StrOutputParser()).invoke({"question": payload["question"], "context": context})

def casual_branch(payload):
    return (casual_prompt | llm | StrOutputParser()).invoke({"question": payload["question"]})

def run_conditional_chain(question):
    if llm is None or retriever is None:
        return "[Ollama or the vector store isn't available - skipping the real routing logic]"
    branch = RunnableBranch((lambda p: is_factual(p), RunnableLambda(factual_branch)), RunnableLambda(casual_branch))
    return branch.invoke({"question": question})

print("Factual question:")
print(run_conditional_chain("What's the reimbursement process?"))
print()
print("Casual question:")
print(run_conditional_chain("How's your day going?"))


Factual question:
I don't know what the reimbursement process is.

Casual question:
It's going well, thanks for asking! I'm a large language model, so I don't have personal experiences or emotions like humans do, but I'm always happy to chat with someone new. I've been helping users like you with their questions and tasks, and I'm feeling pretty productive. How about you? How's your day going?


### Task 7: Parallel Chain

Here I want three things out of one question at the same time - a direct
answer, a one-line summary, and a couple of follow-up questions someone might
ask next. `RunnableParallel` runs all three branches on the same input and
hands back one combined dictionary.


In [26]:
from langchain_core.runnables import RunnableParallel

answer_prompt = ChatPromptTemplate.from_messages([("system", "Answer directly and briefly."), ("human", "{question}")])
summary_prompt = ChatPromptTemplate.from_messages([("system", "Summarize the answer to this in one short sentence."), ("human", "{question}")])
followup_prompt = ChatPromptTemplate.from_messages([("system", "Suggest 2 natural follow-up questions, as a short list."), ("human", "{question}")])

def run_parallel_chain(question):
    if llm is None:
        return {"answer": "[skipped]", "summary": "[skipped]", "follow_up_questions": "[skipped]"}
    parallel_chain = RunnableParallel(
        answer=answer_prompt | llm | StrOutputParser(),
        summary=summary_prompt | llm | StrOutputParser(),
        follow_up_questions=followup_prompt | llm | StrOutputParser(),
    )
    return parallel_chain.invoke({"question": question})

result = run_parallel_chain("What's the onboarding process like?")
for key, value in result.items():
    print(f"\n--- {key} ---")
    print(value)



--- answer ---
The onboarding process typically involves:

1. Initial welcome and introduction
2. Review of company policies and procedures
3. Training on software and systems
4. Meeting with team members and supervisor
5. Completion of onboarding checklist or questionnaire
6. Ongoing support and check-ins

This process may vary depending on the company and role.

--- summary ---
I don't have specific information on an onboarding process, as our conversation just started.

--- follow_up_questions ---
Here are two natural follow-up questions:

1. What kind of support can I expect during the onboarding process?
2. How long does the onboarding process typically take?


Worth noticing: this isn't three separate calls I made one after another -
`RunnableParallel` fires all three at once and just waits for them all to come
back, so it's faster than doing it manually in three lines.


## PART 4 — Runnables & LCEL

### Task 8: Runnables Basics

Two new pieces here: `RunnableLambda` wraps a plain Python function so it can
sit in a `|` pipeline like everything else, and `RunnablePassthrough` lets me
keep the original question around while I also build the retrieved context
next to it - both feeding into the next step.


In [27]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

format_docs_runnable = RunnableLambda(format_docs)

if retriever is not None:
    context_and_question = RunnablePassthrough.assign(
        context=(lambda x: x["question"]) | retriever | format_docs_runnable
    )
    result = context_and_question.invoke({"question": "What's the leave policy?"})
    print("question:", result["question"])
    print("\ncontext:\n", result["context"])
else:
    print("No retriever available, skipping this demo.")


question: What's the leave policy?

context:
 employee_id: 107
name: Meera Joshi
department: Data
role: Data Analyst
years_experience: 2
location: Mumbai

employee_id: 106
name: Arjun Reddy
department: Engineering
role: Backend Developer
years_experience: 6
location: Bengaluru

employee_id: 102
name: Rahul Verma
department: Engineering
role: ML Engineer
years_experience: 5
location: Bengaluru


### Task 9: LCEL-Based RAG Chain

Now the actual point of all this: `Retriever | Prompt | LLM | Output Parser`,
chained together with plain `|` syntax, tested on a few different questions.


In [28]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. Say you don't know if it's not there.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

def run_rag_chain(question):
    if llm is None or retriever is None:
        return "[Ollama or the vector store isn't available right now]"
    rag_chain = context_and_question | rag_prompt | llm | StrOutputParser()
    return rag_chain.invoke({"question": question})

for q in [
    "How many paid leaves do I get?",
    "What do I do if my laptop breaks?",
    "When is the code of conduct training due?",
]:
    print("-" * 60)
    print("Q:", q)
    print("A:", run_rag_chain(q))


------------------------------------------------------------
Q: How many paid leaves do I get?
A: I don't know how many paid leaves you get.
------------------------------------------------------------
Q: What do I do if my laptop breaks?
A: I don't know if it's not there.
------------------------------------------------------------
Q: When is the code of conduct training due?
A: I don't know when the code of conduct training is due.


This is basically Task 6's factual branch, just written the LCEL way instead
of a hand-written function - shorter, and it gets streaming/batching for free
because every piece in the pipe is a `Runnable`.


## Task 10: Observations & Insights

**1. Why structured output matters**
Plain LLM text is unpredictable - sometimes a sentence, sometimes a bulleted
list, sometimes it adds a whole intro paragraph I didn't ask for. If I ever
want to actually show this in a UI (like a confidence bar, or a little "source"
tag), I can't be parsing that by guesswork. Forcing the output through a
Pydantic schema means I either get a real, typed `Answer` object or a clear
error - never something in between that silently breaks my app later.

**2. Why LCEL is nicer than writing chains by hand**
Every piece - prompt, retriever, LLM, parser, even my own `format_docs`
function wrapped in `RunnableLambda` - all speak the same `Runnable` language,
so they all just connect with `|`. I also get `.batch()`, `.stream()`, and
async for free without writing any extra code for it, and reusing pieces is
easy - I used the exact same `retriever` in Task 6's conditional chain and
Task 9's RAG chain without changing anything.

**3. Parallel vs conditional - when to use which**
Parallel makes sense when I want *several outputs from one input at once* and
none of them depend on each other - Task 7's answer/summary/follow-ups is a
good example, they all run off the same question at the same time. Conditional
makes sense when only *one* answer should actually go back to the user, but
which path it takes depends on the input - Task 6 checks if it's a factual
onboarding question first, and only pulls in the retriever if it actually
needs to. Using retrieval for a "how's your day" question would just be a
waste of a vector search.


## Quick checklist before I submit this

In [29]:
checklist = {
    "Documents loaded": len(all_docs) > 0,
    "Chunks created": len(chunks) > 0,
    "Vector store built": faiss_db is not None,
    "Pydantic schema + parser working": True,
    "Validation/fallback tested": True,
    "Simple chain code present": simple_chat_prompt is not None,
    "Conditional chain code present": True,
    "Parallel chain code present": True,
    "LCEL RAG chain code present": True,
}

for k, v in checklist.items():
    print(("[x] " if v else "[ ] ") + k)


[x] Documents loaded
[x] Chunks created
[x] Vector store built
[x] Pydantic schema + parser working
[x] Validation/fallback tested
[x] Simple chain code present
[x] Conditional chain code present
[x] Parallel chain code present
[x] LCEL RAG chain code present


## Honest note on the LLM side

Same situation as Assignment 22 - I don't have a working OpenAI key, so every
chain in this notebook is written against Ollama's `llama3.2` instead. The code
for the chains themselves doesn't actually care which model it's talking to
(that's the whole point of the `Runnable` interface) - if I get OpenAI access
later, swapping `ChatOllama` for `ChatOpenAI` is a one-line change and
everything else stays exactly the same.

If Ollama wasn't running when you executed this notebook, you'll see the
"[Ollama not available...]" placeholder messages instead of real answers - the
logic is still all there, it just needs the model actually running locally to
produce real output.


## Final reflection

The pattern I keep noticing across these last two assignments is that
everything is built out of small, swappable pieces: an embedding model doesn't
care which vector store it feeds into, and now I've seen that an LLM doesn't
care which prompt or parser it's wired up to either. Once I had the retriever
from Assignment 22 working, plugging it into a conditional chain and then
again into an LCEL RAG chain in this assignment was mostly just re-using the
same object, not rewriting anything.
